# Part 1: E-Commerce Customer Segmentation using RFM and K-Means Clustering

This notebook performs an end-to-end customer segmentation analysis for the e-commerce dataset using RFM modeling and K-Means clustering. The goal is to identify valuable customer groups and deliver business recommendations.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
os.makedirs('outputs', exist_ok=True)

## Load Dataset

Load the Part 1 e-commerce customer segmentation dataset from the existing `Data/` folder in this workspace.

In [ ]:
data_path = os.path.join('Data', 'part_1_ecommerce_customer_segmentation.csv')
df = pd.read_csv(data_path)
print('Dataset loaded from:', data_path)
print('Number of rows:', len(df))
print('Number of columns:', len(df.columns))
df.head(10)

## Data Understanding

### Dataset structure and meaning
- `InvoiceNo`: unique order identifier. Invoice codes that start with `C` indicate cancellations.
- `StockCode`: item identifier.
- `Description`: product description.
- `Category`: product category.
- `Quantity`: number of units sold or returned.
- `InvoiceDate`: timestamp of the transaction.
- `UnitPrice`: price per unit in the transaction currency.
- `CustomerID`: unique customer identifier.
- `Country`: customer country.

Each row represents one line item in a customer's invoice. The business type is e-commerce retail, selling consumer products across categories such as Electronics, Home, Beauty, and Apparel.

### What analysis is possible
- Customer segmentation by purchase recency, frequency, and monetary value.
- Identification of high-value customers and promising geographic markets.
- Product performance across quantity and revenue.

### What analysis is not possible
- We cannot infer customer demographics such as age or gender.
- We cannot directly measure marketing channel performance or campaign attribution.
- This dataset lacks session-level or browsing behavior data.

## Data Cleaning

We will clean the dataset carefully and explain each step so the final customer segmentation is reliable.

In [ ]:
initial_shape = df.shape
missing_before = df.isna().sum()
print('Initial shape:', initial_shape)
print('Missing values before cleaning:')
print(missing_before)

# Remove cancelled invoices because these are returns or voided orders
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove rows without CustomerID or Description
# CustomerID is needed for segmentation, Description is necessary for product-level analysis.
df = df.dropna(subset=['CustomerID', 'Description'])

# Convert InvoiceDate to datetime for recency and time-based analysis
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Remove invalid transaction rows with zero or negative quantity or unit price
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Remove exact duplicate rows from the dataset
df = df.drop_duplicates()

cleaned_shape = df.shape
missing_after = df.isna().sum()
print('Cleaned shape:', cleaned_shape)
print('Missing values after cleaning:')
print(missing_after)

# Create a revenue column for line-item value
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df.head()

### Cleaning Explanation
- Cancelled invoices were removed because they represent returns and distort customer value calculations.
- Rows missing `CustomerID` or `Description` were dropped because segmentation requires a known customer and product context.
- Negative or zero values in `Quantity` and `UnitPrice` were removed because they do not represent valid sales.
- Duplicate rows were removed to prevent repeated counting in customer metrics.
- `InvoiceDate` was converted to datetime to support accurate recency and time-series analysis.

## Feature Engineering

We derive customer-level features that are useful for segmentation and business interpretation.

In [ ]:
customer_features = df.groupby('CustomerID').agg({
    'Revenue': 'sum',
    'InvoiceNo': pd.Series.nunique,
    'Quantity': 'sum',
    'StockCode': pd.Series.nunique,
    'Country': 'first'
    }).rename(columns={
    'Revenue': 'TotalRevenue',
    'InvoiceNo': 'TotalPurchases',
    'Quantity': 'TotalQuantity',
    'StockCode': 'UniqueProducts',
    'Country': 'Country'
})

# Calculate average order value per customer
customer_features['AverageOrderValue'] = customer_features['TotalRevenue'] / customer_features['TotalPurchases']
customer_features = customer_features.reset_index()
customer_features.head()

### Feature definitions
- `TotalRevenue`: total spend by the customer.
- `TotalPurchases`: number of unique invoices placed by the customer.
- `TotalQuantity`: total number of items purchased.
- `AverageOrderValue`: average revenue per invoice.
- `UniqueProducts`: count of unique products purchased.
- `Country`: primary customer country.

## Build the RFM Table

Recency, Frequency, and Monetary values form the basis of customer segmentation.

In [ ]:
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'InvoiceNo': pd.Series.nunique,
    'Revenue': 'sum'
}).rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'Revenue': 'Monetary'
}).reset_index()
rfm['Recency'] = rfm['Recency'].astype(int)
rfm['Monetary'] = rfm['Monetary'].round(2)
rfm = rfm.merge(customer_features[['CustomerID', 'Country']], on='CustomerID', how='left')
print('Reference date for recency:', analysis_date.date())
rfm.head()

### RFM column meaning
- `Recency`: how many days since the customer's last purchase. Lower is better.
- `Frequency`: number of unique purchase events. Higher indicates loyalty.
- `Monetary`: total spend in the dataset. Higher indicates more valuable customers.

## Exploratory Data Analysis (EDA)

We explore geography, products, order values, and customer purchase behavior.

In [ ]:
# Top countries by revenue
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(data=country_revenue.head(10), x='Revenue', y='Country', palette='viridis')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig('outputs/top_countries_by_revenue.png')
plt.show()

# Top products by quantity and revenue
top_products_qty = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10).reset_index()
top_products_rev = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(data=top_products_qty, x='Quantity', y='Description', palette='rocket')
plt.title('Top 10 Products by Quantity Sold')
plt.xlabel('Total Quantity')
plt.ylabel('Product Description')
plt.tight_layout()
plt.savefig('outputs/top_products_by_quantity.png')
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(data=top_products_rev, x='Revenue', y='Description', palette='magma')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Revenue')
plt.ylabel('Product Description')
plt.tight_layout()
plt.savefig('outputs/top_products_by_revenue.png')
plt.show()

# Distribution of order value and purchase frequency
invoice_values = df.groupby('InvoiceNo')['Revenue'].sum()
plt.figure(figsize=(12, 5))
sns.histplot(invoice_values, bins=40, kde=True, color='teal')
plt.title('Distribution of Order Value')
plt.xlabel('Order Value')
plt.ylabel('Count of Invoices')
plt.tight_layout()
plt.savefig('outputs/order_value_distribution.png')
plt.show()

plt.figure(figsize=(12, 5))
sns.histplot(rfm['Frequency'], bins=30, kde=False, color='navy')
plt.title('Distribution of Purchase Frequency by Customer')
plt.xlabel('Frequency (unique invoices)')
plt.ylabel('Customer Count')
plt.tight_layout()
plt.savefig('outputs/purchase_frequency_distribution.png')
plt.show()

# Outlier detection for monetary and recency
plt.figure(figsize=(12, 5))
sns.boxplot(x=rfm['Monetary'], color='lightgreen')
plt.title('Outlier Detection: Monetary Value')
plt.xlabel('Monetary Value')
plt.tight_layout()
plt.savefig('outputs/monetary_boxplot.png')
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(x=rfm['Recency'], color='lightcoral')
plt.title('Outlier Detection: Recency')
plt.xlabel('Recency (days)')
plt.tight_layout()
plt.savefig('outputs/recency_boxplot.png')
plt.show()

# Identify high-value customers
high_value_customers = rfm.sort_values(by='Monetary', ascending=False).head(10)
high_value_customers[['CustomerID', 'Country', 'Recency', 'Frequency', 'Monetary']]

### EDA Interpretations
- The top revenue countries show where the strongest markets are, enabling targeted growth in those regions.
- The highest quantity and revenue products indicate best-selling items and best contributors to sales.
- The order value distribution helps us understand how common large vs small orders are.
- Frequency distribution reveals if most customers are one-time or repeat buyers.
- Boxplots identify extreme spenders or long-unused customers that require separate treatment.
- High-value customer profiles are the best targets for loyalty and retention strategies.

## K-Means Clustering on RFM Features

We scale the RFM features, determine the best number of clusters with the elbow method, and create customer segments.

In [ ]:
rfm_features = rfm[['Recency', 'Frequency', 'Monetary']].copy()
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

inertia = []
k_values = range(2, 8)
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(rfm_scaled)
    inertia.append(model.inertia_)

plt.figure(figsize=(10, 5))
sns.lineplot(x=list(k_values), y=inertia, marker='o', color='darkblue')
plt.title('Elbow Method for K-Means Clustering')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.xticks(list(k_values))
plt.tight_layout()
plt.savefig('outputs/kmeans_elbow_plot.png')
plt.show()

# Using 4 clusters based on business segmentation needs and elbow behavior
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
rfm['Cluster'] = rfm['Cluster'].astype(int)
silhouette_avg = silhouette_score(rfm_scaled, rfm['Cluster'])
print('Silhouette score for k=4:', round(silhouette_avg, 4))
rfm.head()

### Cluster interpretation preparation
We now examine cluster means to identify customer segments and assign intuitive labels.

In [ ]:
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'Count'}).round(2).reset_index()
cluster_summary = cluster_summary.sort_values(by=['Monetary', 'Frequency'], ascending=[False, False])
cluster_summary

In [ ]:
# Assign descriptive segment names using cluster patterns
cluster_labels = {}
for _, row in cluster_summary.iterrows():
    cluster = int(row['Cluster'])
    if row['Monetary'] > rfm['Monetary'].mean() and row['Recency'] < rfm['Recency'].mean() and row['Frequency'] > rfm['Frequency'].mean():
        cluster_labels[cluster] = 'High Value'
    elif row['Monetary'] > rfm['Monetary'].mean() and row['Recency'] > rfm['Recency'].mean():
        cluster_labels[cluster] = 'At Risk'
    elif row['Frequency'] > rfm['Frequency'].mean():
        cluster_labels[cluster] = 'Loyal'
    else:
        cluster_labels[cluster] = 'Occasional'

rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
cluster_summary['Segment'] = cluster_summary['Cluster'].map(cluster_labels)
cluster_summary

## Cluster Interpretation

We review the behavioral profile of each customer segment to guide business actions.

In [ ]:
segment_profile = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'CustomerCount'}).round(2).reset_index()
segment_profile

### Segment descriptions
- **High Value**: Customers with high spend, frequent purchases, and recent engagement. These are the best customers to retain and reward.
- **At Risk**: Customers who have spent well but have not purchased recently. They should be targeted with reactivation campaigns.
- **Loyal**: Customers who buy regularly and contribute steady revenue. These customers respond well to loyalty programs.
- **Occasional**: Low frequency and lower spend customers. They are a growth opportunity through promotions or product recommendations.

## Business Recommendations

The final recommendations are tailored to each customer segment.

In [ ]:
recommendations = [
    {'Segment': 'High Value', 'Recommendation': 'Provide premium rewards, early access to launches, and personalized service to reinforce loyalty.'},
    {'Segment': 'At Risk', 'Recommendation': 'Use win-back emails, exclusive discounts, and reminders to recover customers with high past value.'},
    {'Segment': 'Loyal', 'Recommendation': 'Offer loyalty points or subscription bundles that reward repeat purchases and increase order size.'},
    {'Segment': 'Occasional', 'Recommendation': 'Deploy targeted promotions, product recommendations, and low-risk offers to convert occasional buyers into repeat customers.'},
]
recommendation_df = pd.DataFrame(recommendations)
recommendation_df

### Final summary
- We created a clean and customer-centric dataset for segmentation.
- RFM provided a strong foundation for identifying behavior patterns.
- K-Means clusters were used to define business segments with clear action plans.
- The notebook includes saved visualizations in the `outputs/` folder for submission and presentation.